Imports e conexão

In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

# Caminho para o banco
BASE_DIR = Path.cwd()  # Pega a pasta do arquivo atual
ROOT_DIR = BASE_DIR.parent  # Nubank/
db_path = ROOT_DIR / "data" / "nubank.db"

def run_query(sql):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df

### Reclamações

Reclamações por categoria

In [3]:
sql = """
SELECT Categoria_Primaria, COUNT(*) AS total
FROM reclamacoes_com_insights
GROUP BY Categoria_Primaria
ORDER BY total DESC;
"""
df_cat = run_query(sql)
df_cat

,Categoria_Primaria,total
0,Problemas com Cartão,39
1,Cobrança Indevida,22
2,Outros,16
3,PIX & Transferências,12
4,Bloqueio & Acesso,10
5,Atendimento & Suporte,10
6,Empréstimo & Crédito,9
7,Instabilidade do Aplicativo,3
8,Fraude & Vazamento de Dados,1


Distribuição por severidade

In [4]:
sql = """
SELECT Severidade_Estimada, COUNT(*) AS total
FROM reclamacoes_com_insights
GROUP BY Severidade_Estimada;
"""
df_sev = run_query(sql)
df_sev

,Severidade_Estimada,total
0,Alta,4
1,Baixa,45
2,Média,73


Categorias mais críticas

In [5]:
sql = """
SELECT Categoria_Primaria,
       SUM(CASE WHEN Severidade_Estimada='Alta' THEN 1 ELSE 0 END)*1.0/COUNT(*) AS proporcao_alta
FROM reclamacoes_com_insights
GROUP BY Categoria_Primaria
ORDER BY proporcao_alta DESC;
"""
df_prop = run_query(sql)
df_prop

,Categoria_Primaria,proporcao_alta
0,Atendimento & Suporte,0.200000
1,Problemas com Cartão,0.051282
2,PIX & Transferências,0.000000
3,Outros,0.000000
4,Instabilidade do Aplicativo,0.000000
5,Fraude & Vazamento de Dados,0.000000
6,Empréstimo & Crédito,0.000000
7,Cobrança Indevida,0.000000
8,Bloqueio & Acesso,0.000000


### Financeiro

Evolução do Capital Principal

In [11]:
sql = """
SELECT periodo, valor
FROM historico_capital
WHERE descricao='Capital Principal'
  AND periodo IS NOT NULL
ORDER BY periodo;
"""
df_capital = run_query(sql)
df_capital.head()

,periodo,valor
0,2021-06-30,1.648693e+09
1,2021-12-31,2.605154e+09
2,2022-03-22,2.886186e+09
3,2022-03-22,2.886186e+09
4,2022-03-31,2.886186e+09


Índice de Basileia vs ICP

In [12]:
sql = """
SELECT periodo, descricao, valor
FROM historico_capital
WHERE descricao IN ('Índice de Basileia','Índice de Capital Principal (ICP)')
    AND periodo IS NOT NULL
ORDER BY periodo;
"""
df_indices = run_query(sql)
df_indices.head()

,periodo,descricao,valor
0,2021-06-30,Índice de Capital Principal (ICP),0.354631
1,2021-12-31,Índice de Capital Principal (ICP),0.217871
2,2022-03-22,Índice de Capital Principal (ICP),0.175000
3,2022-03-22,Índice de Basileia,0.181400
4,2022-03-22,Índice de Capital Principal (ICP),0.175000


Capital vs RWA

In [8]:
sql = """
SELECT h1.periodo, h1.valor AS capital, h2.valor AS rwa
FROM historico_capital h1
JOIN historico_capital h2 ON h1.periodo=h2.periodo
WHERE h1.descricao='Capital Principal' AND h2.descricao='RWA total'
ORDER BY h1.periodo;
"""
df_comp = run_query(sql)
df_comp.head()

,periodo,capital,rwa
0,2022-03-22,2.886186e+09,1.649349e+10
1,2022-03-22,2.886186e+09,1.649349e+10
2,2022-03-22,2.886186e+09,1.649349e+10
3,2022-03-22,2.886186e+09,1.649349e+10
4,2022-03-31,2.886186e+09,3.260781e+09


Margem excedente média por ano

In [9]:
sql = """
SELECT strftime('%Y', periodo) AS ano, AVG(valor) AS margem_media
FROM historico_capital
WHERE descricao='Margem excedente de Capital Principal (%)'
GROUP BY ano
ORDER BY ano;
"""
df_margin = run_query(sql)
df_margin

,ano,margem_media
0,NaN,0.132993
1,2022,0.127875
2,2023,0.062317
3,2024,0.073350
4,2025,0.060520
